In [9]:
! pip install BeautifulSoup4

from bs4 import BeautifulSoup
# import requests
import urllib.request
import pandas as pd
import datetime


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
#[CODE 1]
def melon_store(result):
    for keyword in range(1,9):
        melon_url = 'https://www.melon.com/search/song/index.htm?q=%s&section=&searchGnbYn=Y&kkoSpl=N&kkoDpType=' % keyword
        print(melon_url)
        
        html = urllib.request.urlopen(melon_url)
        soupmelon = BeautifulSoup(html, 'html.parser')
        tag_tbody = soupmelon.find('tbody')
        tag_tr = tag_tbody.find('tr')
        tag_td = tag_tr.find('td')
        tag_div = tag_td.find('div', class_="ellipsis")
        for store in tag_div:
            if len(store) == 0:
                break  #안정성 위해 추가해줌
            
            title = store.find("a", class_="fc_gray").get_text(strip = True)
            
            
            result.append([title])
    return

In [11]:
#[CODE 0]
def main():
    result = []
    print('Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>')
    melon_store(result)   #[CODE 1] 호출 
    melon_table = pd.DataFrame(result, columns=('title'))
    melon_table.to_csv('melonCrawl.csv', encoding='cp949', mode='w', index=True)
    print('melonCrawl.csv  파일저장 완료 >>>>>>>>>>>>>>>>>')
       
if __name__ == '__main__':
     main()

Crawling >>>>>>>>>>>>>>>>>>>>>>>>>>
https://www.melon.com/search/song/index.htm?q=1&section=&searchGnbYn=Y&kkoSpl=N&kkoDpType=


HTTPError: HTTP Error 406: Not Acceptable

In [14]:
import requests
from bs4 import BeautifulSoup
import urllib.parse
import time

# 406 방지 + 브라우저 위장 User-Agent
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.melon.com/"
}

def search_melon(keyword):
    """멜론 검색 결과에서 노래 리스트 반환"""
    query = urllib.parse.quote(keyword)
    url = f"https://www.melon.com/search/song/index.htm?q={query}&section=&searchGnbYn=Y&kkoSpl=N&kkoDpType="

    res = requests.get(url, headers=HEADERS)
    if res.status_code == 406:
        time.sleep(1)
        res = requests.get(url, headers=HEADERS)

    soup = BeautifulSoup(res.text, "html.parser")

    # HTML 구조 최신 버전 반영
    table = soup.select_one("form#frm_searchSong table tbody")
    if not table:
        print("❌ 검색 결과를 찾지 못했습니다.")
        return []

    songs = []
    for row in table.select("tr"):
        title_tag = row.select_one("div.ellipsis.rank01 a")
        artist_tag = row.select_one("div.ellipsis.rank02 span a")

        if not title_tag or not artist_tag:
            continue

        song_id = title_tag["href"].split("'")[1]  # 'javascript:melon.play.playSong('100000', 'songId')'
        title = title_tag.text.strip()
        artist = artist_tag.text.strip()

        songs.append({
            "id": song_id,
            "title": title,
            "artist": artist
        })

    return songs


def get_melon_lyrics(song_id):
    """곡 ID로 가사 페이지 파싱"""
    url = f"https://www.melon.com/song/detail.htm?songId={song_id}"

    res = requests.get(url, headers=HEADERS)
    if res.status_code == 406:
        time.sleep(1)
        res = requests.get(url, headers=HEADERS)

    soup = BeautifulSoup(res.text, "html.parser")

    lyric_box = soup.select_one("div.lyric")
    if lyric_box:
        return lyric_box.get_text("\n").strip()

    return "가사를 찾을 수 없습니다."


# -----------------------------
# 사용 예시(너 코드 형태 유지)
# -----------------------------
def main():
    keyword = input("검색어 입력: ")
    results = search_melon(keyword)

    if not results:
        print("검색 결과 없음")
    else:
        for idx, s in enumerate(results, start=1):
            print(f"{idx}. {s['title']} - {s['artist']}")

        num = int(input("가사 볼 곡 번호 (여러 개면 콤마로 입력 예: 1,3): "))

        # 여러 곡 선택 가능하도록 처리
        selected_indexes = [int(x.strip()) for x in str(num).split(",")]

        final_data = []
        for idx in selected_indexes:
            song = results[idx - 1]
            lyrics = get_melon_lyrics(song["id"])
            print(f"\n====== {song['title']} 가사 ======")
            print(lyrics)

            # CSV 저장용 구조 추가
            final_data.append({
                "title": song["title"],
                "artist": song["artist"],
                "id": song["id"],
                "lyrics": lyrics
            })

        # 결과 CSV로 저장
        save_to_csv(final_data)


def save_to_csv(filename):
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["Song ID", "Title", "Artist", "Lyrics"])

        # for s in songs:
        #     if selected_song and s["id"] == selected_song["id"]:
        #         writer.writerow([s["id"], s["title"], s["artist"], lyrics])
        #     else:
        #         writer.writerow([s["id"], s["title"], s["artist"], ""])

    print(f"CSV 저장 완료: {filename}")
    
if __name__ == '__main__':
     main()

ModuleNotFoundError: No module named 'requests'